# Tutorial 3: Integrating mouse embryo slices

This tutorial integrates four sagittal mouse embryo sections with **SpaRMA**. It starts from section-level AnnData files, constructs within-section spatial graphs, trains the two-stage model, and performs clustering and visualization.

## 1. Preparation

Place one AnnData file for each section under `Data/Mouse_Embryo/`. Every file must contain an expression matrix and `obsm['spatial']`.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch

from spaarma import fit, get_preset
from spaarma.graph import spatial_edges

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Load data

In [ ]:
data_root = Path("Data/Mouse_Embryo")
section_ids = ["E9.5_E1S1", "E10.5_E2S1", "E11.5_E1S1", "E12.5_E1S1"]

slices = []
for section_id in section_ids:
    sample = sc.read_h5ad(data_root / f"{section_id}.MOSTA.h5ad")
    if "spatial" not in sample.obsm:
        raise KeyError(f"{section_id} does not contain obsm['spatial']")
    sample.var_names_make_unique()
    if "count" in sample.layers:
        sample.X = sample.layers["count"].copy()
    sample.obs_names = [f"{barcode}_{section_id}" for barcode in sample.obs_names]
    sample.obs["batch_name"] = section_id
    sample.obs["batch_name"] = sample.obs["batch_name"].astype("category")
    slices.append(sample)
    print(section_id, sample.n_obs, "spots", sample.n_vars, "genes")

## 3. Preprocess each slice

Each section is normalized and log transformed. Highly variable genes are then identified within each section, and their shared set is used to assemble the joint training object.

In [ ]:
hvg_sets = []
for sample in slices:
    sc.pp.normalize_total(sample, target_sum=1e4)
    sc.pp.log1p(sample)
    sc.pp.highly_variable_genes(sample, flavor="seurat", n_top_genes=5000)
    hvg_sets.append(set(sample.var_names[sample.var["highly_variable"]]))

shared_genes = sorted(set.intersection(*hvg_sets))
if not shared_genes:
    raise RuntimeError("No shared highly variable genes were found.")
processed = [sample[:, shared_genes].copy() for sample in slices]
adata = ad.concat(processed, join="inner", merge="same", uns_merge="unique")
adata.obs["batch_name"] = adata.obs["batch_name"].astype("category")
print(adata)

## 4. Concatenate the Scanpy objects and spatial networks

In [ ]:
preset = get_preset("mouse_embryo")
batches = adata.obs[preset.batch_key].astype(str).to_numpy()
edge_index = spatial_edges(adata.obsm["spatial"], batches, preset.radius)
print("Directed edges including self-loops:", edge_index.shape[1])

## 5. Run SpaRMA

Stage I performs masked spot reconstruction. Stage II restores clean inputs and performs joint reconstruction and multi-positive attention alignment.

In [ ]:
preset = get_preset("mouse_embryo")
batches = adata.obs[preset.batch_key].astype(str).to_numpy()
edge_index = spatial_edges(adata.obsm["spatial"], batches, preset.radius)
x = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)

embedding, settings, model = fit(x, edge_index, batches, preset.training, device=device)
adata.obsm["SpaRMA"] = embedding
adata.uns["SpaRMA"] = settings
print("Embedding shape:", embedding.shape)

## 6. Clustering

The learned embedding is clustered after optimization. Louvain resolution 0.5 is used here; annotations may be joined later for biological interpretation.

In [ ]:
sc.pp.neighbors(adata, use_rep="SpaRMA")
sc.tl.louvain(adata, resolution=0.5, key_added="SpaRMA_domain", random_state=0)
sc.tl.umap(adata, random_state=0)

## 7. Visualization

In [ ]:
sc.pl.umap(adata, color=["batch_name", "SpaRMA_domain"], wspace=0.35)

fig, axes = plt.subplots(1, len(section_ids), figsize=(4 * len(section_ids), 4))
for ax, section_id in zip(axes, section_ids):
    view = adata[adata.obs["batch_name"] == section_id]
    sc.pl.embedding(view, basis="spatial", color="SpaRMA_domain", title=section_id, ax=ax, show=False, size=12)
plt.tight_layout()